# GST drift mechanism study — pipeline walkthrough (v1)

Companion notebook to [`EXPERIMENT.md`](EXPERIMENT.md). Runs the full chain:

1. Load `gst_drift.yaml` (4 reference structures: 1 PBC bulk + 3 H-passivated cluster dumbbells; the larger Peierls / tetrahedral clusters are deferred to v2 — see EXPERIMENT.md §4.2 / §4.3).
2. **`qm-relax`** the references at their assigned QM backend:
   - `GST_rocksalt` → **Quantum ESPRESSO**, PBE, plane-wave.
   - `Te2_wrong`, `Ge2_dumbbell`, `Sb2_dumbbell` → **PySCF**, B3LYP/def2-svp + def2-ecp + density fitting.

   PyField dispatches per-structure based on the `qm_code:` override.
3. **`make-scan`** → 40 perturbed structures across 7 scans (4 PBC strain scans + 3 homopolar-bond stretch scans).
4. **`qm-prep`** runs the constrained QM relaxes that populate every `target: { from: dft }`.
5. **CMA-ES** refit on 28 trainable parameters (see `params_GST` and EXPERIMENT.md §5.5).
6. Before/after `cost_breakdown` to read off which targets the optimiser pulled down.

**Heavy demo. Workstation, not CI.** Cold-cache QM is ~5–8 hours on 8 MPI cores (most of it the QE bulk strain scans). Without MPI it's ~30 hours — set this *before* launching:

```
export ESPRESSO_COMMAND='mpirun -np 8 pw.x -in PREFIX.pwi > PREFIX.pwo'
```

Requirements:
- `pip install -e .[dev,cma]` (PySCF, geomeTRIC, ASE, CMA, matplotlib).
- `pw.x` on `PATH` (see README §"Setting up Quantum ESPRESSO").
- SSSP UPF pseudopotentials extracted somewhere; `qm.pseudo_dir` in `gst_drift.yaml` points at them (default `/home/ubuntu/qe_pseudos`).
- LAMMPS via `pip install 'lammps[mpi]'`.

**v1 scope honesty.** The two larger / higher-symmetry clusters in the original training plan are deferred:

| Cluster | Why deferred | What we lose |
|---|---|---|
| `GeTe6_Peierls` (37 atoms) | Too expensive for PySCF B3LYP/def2-svp on a single workstation (>30 min per geom-opt step) | Direct probe of Peierls double-well (drift mechanism B). Bulk strain scans on `GST_rocksalt` partially cover this. |
| `GeTe4_tet` (9 atoms, Td) | Perfect Td symmetry → degenerate t₂ orbitals → PySCF SCF can't navigate | Direct 4-fold-vs-6-fold Ge coordination signal. Homopolar Ge–Ge dumbbell partially covers this. |

Both unblock once an ORCA backend (`pyfield.qm.orca_backend`) lands — RIJCOSX hybrid-DFT acceleration and better symmetry handling for heavy-element clusters.

In [ ]:
# Sanity check #0 — make sure the kernel is running the current pyfield code,
# not a stale module from before per-structure backend dispatch landed.
# If this assert fires, restart the kernel (Kernel → Restart) and re-run.
import importlib
import pyfield.qm.base
import pyfield.qm.prep
importlib.reload(pyfield.qm.base)
importlib.reload(pyfield.qm.prep)
import inspect
_src = inspect.getsource(pyfield.qm.prep.populate_qm)
assert 'backends.for_structure' in _src, (
    "Stale module in kernel — populate_qm is the OLD version that uses one "
    "backend for all structures (so PySCF gets called on the GST PBC cell "
    "and you'll see a 467-electron / spin-0 mismatch). RESTART THE KERNEL: "
    "Kernel → Restart Kernel, then re-run from cell 1."
)
print('OK — populate_qm has per-structure backend dispatch.')

## 1. Load and inspect the config

In [ ]:
import os
from pathlib import Path
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..')) if os.path.basename(os.getcwd()) == 'gst_drift' else os.getcwd()
os.chdir(ROOT)

from pyfield.config.loader import load_yaml
from pyfield.qm.base import structure_code

CFG_PATH = 'studies/gst_drift/gst_drift.yaml'
cfg = load_yaml(CFG_PATH)
print(f'qm default:    {cfg.qm.code}/{cfg.qm.functional}/{cfg.qm.basis}')
print(f'qm.pseudo_dir: {(cfg.qm.__pydantic_extra__ or {}).get("pseudo_dir")}')
print(f'ESPRESSO_COMMAND env: {os.environ.get("ESPRESSO_COMMAND", "(unset — single-core pw.x)")}')
print()
print(f'structures:    {len(cfg.structures)}')
for n, s in cfg.structures.items():
    code = structure_code(s, fallback=cfg.qm.code)
    extras = s.__pydantic_extra__ or {}
    func = extras.get('qm_functional', cfg.qm.functional)
    mode = '(PBC)' if s.pbc else '(cluster)'
    print(f'               {n:<22} {mode:<10} backend={code:<6} func={func:<6} {len(s.atoms)} atoms')
print(f'scans:         {len(cfg.scans)}')
for s in cfg.scans:
    n = len(s.values) if s.values else s.range[2]
    print(f'               {s.type:<19} ref={s.reference:<18} N={n}  relax={s.relax_method}')
print()
print(f'optimiser:     {cfg.optimizer.method}, generations={cfg.optimizer.generations}, '
      f'parallel={cfg.optimizer.parallel}, processors={cfg.optimizer.processors}')
print(f'trainable:     28 parameters in studies/gst_drift/params_GST')

In [ ]:
# Diagnostic — verify what's in cfg RIGHT AFTER loading. Confirms (a)
# we loaded the right YAML, (b) the pbc / qm_code / qm_functional
# extras survived YAML parsing.
import os
abs_path = os.path.abspath(CFG_PATH)
print(f'CFG_PATH (abs):       {abs_path}')
print(f'file exists:          {os.path.exists(abs_path)}')
print(f'file size:            {os.path.getsize(abs_path) if os.path.exists(abs_path) else "?"} bytes')
print()

ggst = cfg.structures['GST_rocksalt']
extras = ggst.__pydantic_extra__ or {}
print(f'GST_rocksalt:')
print(f'  pbc:                {ggst.pbc}')
print(f'  qm_relax:           {ggst.qm_relax}')
print(f'  qm_relax_cell:      {ggst.qm_relax_cell}   # vc-relax: QE finds the true V_eq')
print(f'  qm_code (extra):    {extras.get("qm_code")}')
print(f'  qm_functional:      {extras.get("qm_functional")}')
print(f'  box (seed):         {tuple(ggst.box)}   # vc-relax will move it')
print(f'  atoms count:        {len(ggst.atoms)}')

if not ggst.pbc:
    print()
    print('!! pbc=False on GST_rocksalt — your YAML is missing `pbc: true`.')
    print('   Open studies/gst_drift/gst_drift.yaml and check the GST_rocksalt block.')
elif extras.get('qm_code') != 'qe':
    print()
    print('!! qm_code is not "qe" on GST_rocksalt — `qm_code: qe` missing or being dropped.')
else:
    print()
    print('OK — bulk reference correctly tagged for QE backend.')

## 2. `qm-relax` — find equilibrium geometries

PyField dispatches each structure to its assigned backend (`qm_code:` per-structure override). PySCF + geomeTRIC handles the molecular clusters; QE + ASE BFGS handles the periodic rocksalt cell.

Wall-clock budget on 8 MPI cores (cold cache):

| Structure | Backend | Atoms | Per-call wall-clock |
|---|---|---|---|
| `GST_rocksalt` | QE / PBE / Γ-only | 18 | ~30–60 min for the **vc-relax** (atoms + cell, internal QE BFGS, `cell_factor: 2.0`) |
| `GeTe6_Peierls` | PySCF / B3LYP | 37 | ~15–25 min |
| `GeTe4_tet` | PySCF / B3LYP | 9 | ~3–5 min |
| `Te2_wrong`, `Ge2_dumbbell`, `Sb2_dumbbell` | PySCF / B3LYP | 4–8 | ~1–2 min each |

**Total cold cache for the references: ~1.5 hours on 8 cores.** All cached afterwards.

The bulk reference uses **`qm_relax_cell: true`** (see `gst_drift.yaml`) so QE
finds its own equilibrium volume for PBE — anchoring the strain ladder symmetrically
around `V_eq` rather than on the literature 6.02 Å seed. Without this the strain
targets become an asymmetric slope that ReaxFF physically can't fit; see
[EXPERIMENT.md §10 (2026-05-09)](EXPERIMENT.md) for the diagnostic.

In [ ]:
import importlib.util
assert importlib.util.find_spec('pyscf') and importlib.util.find_spec('geometric'), \
    'this notebook requires `pip install -e .[dev,cma]` (pulls pyscf + geometric + cma)'
assert importlib.util.find_spec('ase'), 'ASE is required for the QE backend (`pip install ase`)'
import shutil
if not shutil.which('pw.x'):
    raise RuntimeError(
        "pw.x not found on PATH. Install QE (apt install quantum-espresso, or "
        "conda install -c conda-forge qe) — see README §'Setting up Quantum ESPRESSO'."
    )

from pyfield.qm.prep import cfg_to_yaml, relax_structures

relaxed_cfg, journal = relax_structures(cfg)
for action, hit, key in journal:
    tag = '[cache hit ]' if hit else '[running   ]'
    print(f'  {tag} {action}   ({key})')

relaxed_path = Path('studies/gst_drift/gst_drift.relaxed.yaml')
relaxed_path.write_text(cfg_to_yaml(relaxed_cfg))
print(f'\nrelaxed YAML written to {relaxed_path}')

## 3. `make-scan` — expand into 68 perturbed structures

Strain scans on the PBC rocksalt cell (22 points), the Peierls double-well (9 points), bond / angle scans on each cluster (37 points). xyz snapshots get dumped under `runs/scan_xyz/` for inspection.

In [ ]:
from pyfield.scans import expand_scans

scanned_cfg, summary = expand_scans(
    relaxed_cfg, xyz_dir=Path('studies/gst_drift/runs/scan_xyz')
)
for line in summary:
    print(' ', line)

scanned_path = Path('studies/gst_drift/gst_drift.scanned.yaml')
scanned_path.write_text(cfg_to_yaml(scanned_cfg))
print()
print(f'structures: {len(scanned_cfg.structures)}')
print(f'simulations: {len(scanned_cfg.simulations)}')
print(f'targets: {len(scanned_cfg.targets)}')
print(f'\nscanned YAML → {scanned_path}')

## 4. Visualise a homopolar bond-stretch scan

Animate the Te–Te wrong-bond fragment as it dissociates. (The headline Peierls displacement scan is deferred to v2 — see intro for why.)

In [ ]:
import matplotlib
matplotlib.use('Agg')
from pyfield.viz import animate_xyz_dir

animate_xyz_dir(
    'studies/gst_drift/runs/scan_xyz',
    pattern='Te2_d_*.xyz',
    interval_ms=400,
    title='Te–Te homopolar bond stretch',
)

## 5. `qm-prep` — populate every `target: { from: dft }`

This is the bulk of the cold-cache time. 68 constrained QM relaxes (or single-points for `relax_method: rigid` scans), dispatched per-structure:

- **PBC strain scans on `GST_rocksalt`** — QE / PBE. ASE BFGS holds the lattice fixed at each strained value and relaxes atoms inside. ~25–35 min per scan point at the demo settings.
- **Cluster scans (Peierls, bond-stretch, angle-bend, homopolar bonds)** — PySCF / B3LYP/def2-svp + def2-ecp. Constrained by `geometric_solver` `$set` blocks. ~3–10 min per scan point.

Plan **~6–8 hours cold** on 8 MPI cores (Quantum ESPRESSO MPI = the bottleneck). Subsequent runs hit the cache and finish in seconds.

In [ ]:
# Diagnostic — show every structure in scanned_cfg, so we can see exactly
# what populate_qm is about to dispatch to which backend. Empty PBC list
# means the bulk wasn't expanded into the cfg correctly.
from pyfield.qm.base import structure_code

print(f'scanned_cfg.structures count: {len(scanned_cfg.structures)}')
print(f'scanned_cfg.simulations count: {len(scanned_cfg.simulations)}')
print(f'scanned_cfg.targets count:    {len(scanned_cfg.targets)}')
print()
print(f"{'structure':<22} {'pbc':<6} {'qm_relax':<10} {'vc':<5} {'qm_code':<8} {'qm_func':<8} atoms")
for n, s in scanned_cfg.structures.items():
    extras = s.__pydantic_extra__ or {}
    code = structure_code(s, fallback=scanned_cfg.qm.code)
    print(f'  {n:<22} {str(s.pbc):<6} {str(s.qm_relax):<10} '
          f'{str(s.qm_relax_cell):<5} '
          f'{code:<8} {str(extras.get("qm_functional")):<8} {len(s.atoms or [])}')

In [ ]:
from pyfield.qm.prep import populate_qm

populated, journal = populate_qm(scanned_cfg)
ran = sum(1 for a, hit, _ in journal if not hit and 'reuse' not in a)
cached = sum(1 for _, hit, _ in journal if hit)
reused = sum(1 for a, _, _ in journal if 'reuse' in a)
print(f'qm-prep: {ran} ran fresh, {cached} cache hits, {reused} reused-relax-energy, {len(journal)} total')

populated_path = Path('studies/gst_drift/gst_drift.populated.yaml')
populated_path.write_text(cfg_to_yaml(populated))
print(f'\npopulated YAML → {populated_path}')
print('\nDFT ΔE targets (first 12 shown):')
for tgt in populated.targets[:12]:
    extras = tgt.__pydantic_extra__
    sim_id = next(iter(extras['terms']))
    print(f'  {sim_id:<22}  ΔE = {extras["target"]:>10.4f} kcal/mol')
print(f'  ... ({len(populated.targets) - 12} more)')

## 6. CMA-ES refit

2000 generations, 8 worker processes, default popsize (13 children/gen for N=28 trainable parameters). `tqdm` shows live `gen=…, best=…, sigma=…` while it runs. CMA's built-in convergence test (`tolfun`, `tolx`) may stop earlier than 2000 if the cost surface has flattened.

Wall-clock plan: ~20–40 s/generation × 2000 generations ≈ 12–24 h on a workstation. CMA's `sigma` shrinks toward zero as it converges; a sigma below ~1e-3 is the canonical "done" signal.

In [ ]:
from pyfield.io.lammps import preload_libmpi
preload_libmpi()
from pyfield.optimizers import run_optimizer
from pyfield.diagnostics import cost_breakdown


def _print_breakdown(report, head=12):
    print(f'total cost = {report.total_cost:.4f}')
    print('(top per-target residuals shown:)')
    rows = sorted(report.target_reports, key=lambda r: -r.residual)[:head]
    for r in rows:
        sim_id = r.description.split()[0].lstrip('+')[2:]
        target = r.target if r.target is not None else 0
        print(f'  {sim_id:<22}  FF={r.value:>9.4f}  '
              f'target={target:>9.4f}  residual={r.residual:>9.2f}')


print('=' * 70)
print(f'BEFORE {populated.optimizer.method.upper()} — initial seed FF vs B3LYP/def2-svp targets')
print('=' * 70)
before = cost_breakdown(populated)
_print_breakdown(before)

result = run_optimizer(populated)

print()
print('=' * 70)
print(f'AFTER  {populated.optimizer.method.upper()} — best FF written to {result.best_ffield_path}')
print('=' * 70)
after = cost_breakdown(populated, ffield_path=result.best_ffield_path)
_print_breakdown(after)
print()
print(f'final cost: {result.final_cost:.4f}')
print(f'cost trace length: {len(result.cost_trace)}')
print(f'reduction: {before.total_cost:.1f} → {after.total_cost:.1f}  '
      f'({100*(1-after.total_cost/before.total_cost):.1f}% drop)')

## 7. Cost trace

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(result.cost_trace, marker='.', linewidth=0.5)
ax.set_xlabel('CMA generation')
ax.set_ylabel('best-so-far cost')
ax.set_title('GST ReaxFF refit — CMA-ES cost trace')
ax.set_yscale('log')
fig.tight_layout()
fig.savefig('studies/gst_drift/cost_trace.png', dpi=80)

## 8. Next steps

Once the FF is fitted to satisfaction, the production simulations live outside this notebook (plain LAMMPS MD using `studies/gst_drift/runs/sa/bestFF.reax`):

1. Build a 5–10 nm amorphous Ge₂Sb₂Te₅ slab by melt-quench MD (1500 K → 300 K @ 10¹² K/s).
2. Anneal at 300, 350, 400, 425 K for 50–100 ns.
3. Measure the three drift observables (mechanism A / B / C — see EXPERIMENT.md §1).
4. Cross-validate against EXAFS / PDF / drift coefficient (refs in EXPERIMENT.md §11).

Update `EXPERIMENT.md §10` with each milestone.